# `examples/custom_function_sim` 可运行诊断版

这个 Notebook 对应 `examples/custom_function_sim/main.py`。

上一版 Notebook 如果在 Jupyter 中看不到输出，通常是因为 Jupyter 已经提前配置过 logging，导致 `logging.basicConfig(level=logging.INFO)` 不再生效。这个版本做了两点调整：

1. 使用 `logging.basicConfig(..., force=True)` 强制把日志输出到 Notebook；
2. 在关键阶段加入 `print(..., flush=True)`，即使日志被 Jupyter 屏蔽，也能看到运行进度。

运行顺序：从上到下逐个执行代码单元即可。重点观察第 4 节的输出。

## 1. 初始化路径和日志

这一单元只做环境准备：

- 自动推断 faas-sim 项目根目录；
- 把项目根目录加入 `sys.path`；
- 强制配置 logging，使日志在 Notebook 中可见。

In [1]:
import logging
import sys
from pathlib import Path

# 自动寻找项目根目录：向上查找同时包含 sim/ 和 examples/ 的目录。
current_dir = Path.cwd().resolve()
search_roots = [current_dir] + list(current_dir.parents)

PROJECT_ROOT = None
for root in search_roots:
    if (root / "sim").exists() and (root / "examples").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "没有找到 faas-sim 项目根目录。请把 Notebook 放到项目目录内运行，"
        "或先切换 Jupyter 当前工作目录到 faas-sim-master。"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Jupyter 中 basicConfig 可能被已有 handler 拦截，force=True 会重置日志配置。
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

print(f"当前工作目录：{current_dir}", flush=True)
print(f"项目根目录：{PROJECT_ROOT}", flush=True)
print("日志系统已强制配置，后续 faas-sim INFO 日志应能在 Notebook 中显示。", flush=True)

当前工作目录：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master\examples\custom_function_sim
项目根目录：C:\Users\weew12\Downloads\faas-sim-master_内置依赖兼容性检查版\faas-sim-master
日志系统已强制配置，后续 faas-sim INFO 日志应能在 Notebook 中显示。


## 2. 导入依赖

这里复用 `examples/basic/main.py` 中的基础拓扑和基础 Benchmark，只替换函数生命周期模拟器。

In [2]:
import examples.basic.main as basic
import sim.docker as docker

from sim.core import Environment
from sim.faas import (
    FunctionSimulator,
    FunctionReplica,
    FunctionRequest,
    SimulatorFactory,
    FunctionContainer,
)
from sim.faassim import Simulation

logger = logging.getLogger(__name__)

print("依赖导入完成。", flush=True)

依赖导入完成。


## 3. 定义自定义函数模拟器

`CustomSimulatorFactory` 负责告诉 faas-sim：每个函数副本应该使用哪个 `FunctionSimulator`。

`MyFunctionSimulator` 实现五个生命周期阶段：

- `deploy`：镜像拉取；
- `startup`：副本启动；
- `setup`：业务初始化；
- `invoke`：请求执行；
- `teardown`：副本关闭。

In [4]:
class CustomSimulatorFactory(SimulatorFactory):
    """
    自定义函数模拟器工厂。

    faas-sim 在创建函数副本时会调用 create(env, fn)，并把返回的 FunctionSimulator
    绑定到该副本上。这里为了演示机制，所有函数容器都返回 MyFunctionSimulator。
    """

    def __init__(self) -> None:
        super().__init__()

    def create(self, env: Environment, fn: FunctionContainer) -> FunctionSimulator:
        """
        为函数容器创建生命周期模拟器。

        参数：
        - env：仿真环境；
        - fn：函数容器配置，包含镜像和资源请求等信息。

        返回：
        - MyFunctionSimulator：自定义函数生命周期模拟器。
        """
        return MyFunctionSimulator()


class MyFunctionSimulator(FunctionSimulator):
    """
    自定义函数生命周期模拟器。

    当前模拟规则：
    - deploy 阶段调用 docker.pull，模拟镜像拉取；
    - startup 阶段固定等待 10 个仿真时间单位；
    - setup 阶段不额外等待；
    - invoke 阶段根据函数名和节点类型设置不同执行时间；
    - teardown 阶段不额外等待。
    """

    def deploy(self, env: Environment, replica: FunctionReplica):
        """
        模拟函数副本部署阶段。

        docker.pull 会检查目标节点是否已有镜像；如果没有，则通过 Ether 网络模型模拟镜像下载。
        因此该阶段可用于观察镜像大小和网络条件对冷启动部署时间的影响。
        """
        yield from docker.pull(env, replica.container.image, replica.node.ether_node)

    def startup(self, env: Environment, replica: FunctionReplica):
        """
        模拟函数副本启动阶段。

        这里固定等待 10 个仿真时间单位，用来近似容器启动、运行时初始化或 watchdog 启动耗时。
        """
        logger.info(
            "[simtime=%.2f] starting up function replica for function %s",
            env.now,
            replica.function.name,
        )
        yield env.timeout(10)

    def setup(self, env: Environment, replica: FunctionReplica):
        """
        模拟业务初始化阶段。

        当前示例不额外模拟模型加载、连接建立或缓存预热，因此等待时间为 0。
        """
        yield env.timeout(0)

    def invoke(self, env: Environment, replica: FunctionReplica, request: FunctionRequest):
        """
        模拟一次函数调用。

        业务流程：
        1. 输出调用日志；
        2. 按节点 CPU 容量的 10% 登记本次请求的 CPU 占用；
        3. 把请求加入节点 current_requests 集合；
        4. 根据函数名和节点类型等待不同执行时间；
        5. 执行完成后释放 CPU 占用并移除请求。
        """
        logger.info(
            "[simtime=%.2f] invoking function %s on node %s",
            env.now,
            request,
            replica.node.name,
        )

        cpu_millis = replica.node.capacity.cpu_millis * 0.1
        env.resource_state.put_resource(replica, "cpu", cpu_millis)

        node = replica.node
        node.current_requests.add(request)

        if replica.function.name == "python-pi":
            if replica.node.name.startswith("rpi3"):
                yield env.timeout(20)
            else:
                yield env.timeout(2)
        elif replica.function.name == "resnet50-inference":
            yield env.timeout(0.5)
        else:
            yield env.timeout(0)

        env.resource_state.remove_resource(replica, "cpu", cpu_millis)
        node.current_requests.remove(request)

    def teardown(self, env: Environment, replica: FunctionReplica):
        """
        模拟副本关闭阶段。

        当前示例不额外模拟关闭耗时。
        """
        yield env.timeout(0)

print("自定义 SimulatorFactory 和 FunctionSimulator 定义完成。", flush=True)

自定义 SimulatorFactory 和 FunctionSimulator 定义完成。


## 4. 运行仿真

这个单元会真正执行 faas-sim。为了避免“看起来没输出”，这里加入了显式 `print` 阶段提示。

如果这个单元长时间不结束，请先看它停在哪一条日志之后。

In [5]:
def main():
    """
    自定义函数模拟器示例入口。

    返回：
    - sim：运行完成后的 Simulation 对象，后续单元可从 sim.env.metrics 中提取指标。
    """
    print("[1/5] 创建 topology...", flush=True)
    topology = basic.example_topology()
    print(f"[2/5] topology 创建完成，节点数={len(topology.nodes)}", flush=True)

    print("[3/5] 创建 benchmark...", flush=True)
    benchmark = basic.ExampleBenchmark()

    print("[4/5] 创建 Simulation，并替换 simulator factory...", flush=True)
    sim = Simulation(topology, benchmark)
    sim.create_simulator_factory = CustomSimulatorFactory

    print("[5/5] 开始运行 sim.run()...", flush=True)
    sim.run()
    print(f"仿真结束：simtime={sim.env.now:.2f}", flush=True)

    return sim

sim = main()

[1/5] 创建 topology...
[2/5] topology 创建完成，节点数=108
[3/5] 创建 benchmark...
[4/5] 创建 Simulation，并替换 simulator factory...
[5/5] 开始运行 sim.run()...
2026-07-03 21:34:51,133 [INFO] sim.faassim: initializing simulation, benchmark: ExampleBenchmark, topology nodes: 108
2026-07-03 21:34:51,133 [INFO] sim.faassim: starting resource monitor
2026-07-03 21:34:51,133 [INFO] sim.faassim: setting up benchmark
2026-07-03 21:34:51,134 [INFO] examples.basic.main: python-pi-cpu, latest, [ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='arm32'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='x86'), ImageProperties(name='python-pi-cpu', size=58000000, tag='latest', arch='aarch64')]
2026-07-03 21:34:51,134 [INFO] examples.basic.main: resnet50-inference-cpu, latest, [ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='arm32'), ImageProperties(name='resnet50-inference-cpu', size=56000000, tag='latest', arch='x86'), ImageProperties(name

## 5. 查看核心指标

运行结束后，先检查 `invocations`、`schedule`、`replica_deployment`、`flow` 是否有数据。

In [6]:
metric_names = [
    "invocations",
    "schedule",
    "replica_deployment",
    "function_deployment",
    "function_deployment_lifecycle",
    "flow",
    "network",
    "node_utilization",
    "function_utilization",
]

metric_dfs = {}
for name in metric_names:
    try:
        df = sim.env.metrics.extract_dataframe(name)
        metric_dfs[name] = df
        print(f"{name}: {len(df)} 行", flush=True)
    except Exception as exc:
        print(f"{name}: 提取失败，原因：{exc}", flush=True)

invocations: 20 行
schedule: 6 行
replica_deployment: 8 行
function_deployment: 2 行
function_deployment_lifecycle: 2 行
flow: 2 行
network: 0 行
node_utilization: 0 行
function_utilization: 4 行


In [7]:
# 查看函数调用记录。
invocations_df = metric_dfs.get("invocations")
invocations_df.head() if invocations_df is not None else None

,t_wait,t_exec,t_start,memory,function_name,function_image,node,replica_id
time,,,,,,,,
2026-07-03 21:34:51.168249,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_0,2212429260240
2026-07-03 21:34:51.168270,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_0,2212429260240
2026-07-03 21:34:51.168285,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_0,2212429260240
2026-07-03 21:34:51.168299,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_0,2212429260240
2026-07-03 21:34:51.168312,0.0,0.5,11.0,1073741824,resnet50-inference,resnet50-inference-gpu,server_0,2212429260240


In [8]:
# 按函数统计执行时间。
if invocations_df is not None and "function_name" in invocations_df.columns and "t_exec" in invocations_df.columns:
    display(
        invocations_df
        .groupby("function_name")["t_exec"]
        .agg(["count", "mean", "min", "max"])
        .reset_index()
    )
else:
    print("当前 invocations_df 字段：", None if invocations_df is None else list(invocations_df.columns))

,function_name,count,mean,min,max
0,python-pi,10,2.0,2.0,2.0
1,resnet50-inference,10,0.5,0.5,0.5


In [9]:
# 查看调度完成事件。
schedule_df = metric_dfs.get("schedule")
if schedule_df is not None and len(schedule_df) > 0:
    if "value" in schedule_df.columns:
        display(schedule_df[schedule_df["value"] == "finish"].head())
    else:
        display(schedule_df.head())
else:
    print("schedule_df 为空。")

,value,function_name,image,replica_id,node_name,successful
time,,,,,,
2026-07-03 21:34:51.150899,finish,python-pi,python-pi-cpu,2212429697712,server_0,True
2026-07-03 21:34:51.154524,finish,resnet50-inference,resnet50-inference-gpu,2212429260240,server_0,True


In [10]:
# 查看副本部署生命周期记录。
replica_deployment_df = metric_dfs.get("replica_deployment")
replica_deployment_df.head() if replica_deployment_df is not None else None

,value,function_name,node_name,replica_id
time,,,,
2026-07-03 21:34:51.153489,deploy,python-pi,server_0,2212429697712
2026-07-03 21:34:51.155406,deploy,resnet50-inference,server_0,2212429260240
2026-07-03 21:34:51.155940,startup,resnet50-inference,server_0,2212429260240
2026-07-03 21:34:51.156870,startup,python-pi,server_0,2212429697712
2026-07-03 21:34:51.157534,setup,resnet50-inference,server_0,2212429260240


## 6. 如果仍然没有输出

如果第 4 节还是一直运行且没有任何日志，优先检查：

1. 当前 Kernel 是否已经卡住，尝试 Restart Kernel 后从头运行；
2. Notebook 是否在正确项目环境中运行，确认 `PROJECT_ROOT` 是 faas-sim-master；
3. 终端运行能成功而 Notebook 失败时，确认 Jupyter 使用的是同一个 Conda 环境；
4. 在 Notebook 中执行 `import sys; print(sys.executable)`，确认 Python 解释器路径是否和终端一致。